# Cross-City Transfer Learning for Bike-Sharing Demand Forecasting
**TUM Deep Learning & Decision Making — Melis Üresin, Nil Yılmazcan, Sefa Kosova**

This notebook runs the full experiment pipeline:
1. Mount Google Drive and clone the repo
2. Build the hourly panel from the interim data
3. Build supervised samples (windowing)
4. Run all 5 methods across all 4 target-history budgets
5. Plot the transfer-gain curve

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = "https://github.com/melisuresin-oss/bike-sharing-transfer-learning.git"
BRANCH = "feature/data-pipeline"
REPO_DIR = "/content/bike-sharing-transfer-learning"

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Verify the interim data is accessible on Google Drive
import os
from pathlib import Path

DRIVE_INTERIM = Path("/content/drive/MyDrive/bikeshare-transfer/data/interim")
expected = ["stations.parquet", "trips.parquet", "station_status.parquet"]

for f in expected:
    path = DRIVE_INTERIM / f
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        print(f"  {f}: {size_mb:.1f} MB")
    else:
        print(f"  MISSING: {f}")

## 2. Build the Hourly Panel

Reads the filtered interim parquets and builds a per-station, per-hour departure count with a coverage mask. Expected output: ~2,731,800 eligible station-hours across 4 cities.

In [ ]:
!python src/data/build_panel.py --config configs/colab.yaml

## 3. Build Supervised Samples

Converts the panel into windowed (sequence, covariates, target) samples for each city, split and budget. Also fits and saves per-city demand scalers.

In [ ]:
!python src/features/windowing.py --config configs/colab.yaml

## 4. Run the Experiment

Trains and evaluates all 5 methods across all 4 target-history budgets (1 day, 7 days, 30 days, full). Results saved to `results/all_metrics.csv`.

**Expected runtime:** 2–4 hours on CPU, ~30–60 min on GPU.

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
!python src/run_experiment.py --config configs/colab.yaml --output results/all_metrics.csv

## 5. Results

In [ ]:
import pandas as pd

df = pd.read_csv("results/all_metrics.csv")

# Pivot: methods as rows, budgets as columns, MAE as values
pivot = df.pivot_table(index="method", columns="budget", values="mae")
method_order = ["historical_average", "persistence", "target_only_gru", "pooled_gru", "source_pretrained_finetuned_gru"]
pivot = pivot.reindex([m for m in method_order if m in pivot.index])
pivot.columns.name = "Budget (days)"
print("MAE by method and budget:")
display(pivot.round(4))

## 6. Transfer-Gain Curve

In [ ]:
!python src/plot_results.py --results results/all_metrics.csv --out results/transfer_gain_curve.png

from IPython.display import Image
Image("results/transfer_gain_curve.png")

In [ ]:
# Save results to Google Drive for backup
import shutil
from pathlib import Path

drive_results = Path("/content/drive/MyDrive/bikeshare-transfer/results")
drive_results.mkdir(parents=True, exist_ok=True)

shutil.copy("results/all_metrics.csv", drive_results / "all_metrics.csv")
shutil.copy("results/transfer_gain_curve.png", drive_results / "transfer_gain_curve.png")
print(f"Saved results to {drive_results}")